# Human pilot: success versus depth

Task 1 (bowl on stove) by default; 10 trials at each of depths 2, 4, 6, 8, 10 for FC, native FRS, and FRS+adaptor: 150 trials. Run `experiments/human_pilot.sh` first, then set `ROOT` below and Run All.

This uses the same plotting function as `analyze_sweep.ipynb`. The x-axis counts FC guided steps or FRS reversal steps; equal numeric depths are not equal operator authority. Keep each participant/session in its own output directory.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / 'examples/pi05/libero_shared_autonomy/notebooks/analyze.py').exists())
sys.path.insert(0, str(REPO_ROOT / 'examples/pi05/libero_shared_autonomy/notebooks'))
import analyze

ROOT = REPO_ROOT / 'outputs/pi05_libero_human_pilot_today'
DEPTHS = [2, 4, 6, 8, 10]
TRIALS_PER_CONDITION = 10


In [ ]:
runs = analyze.find_runs(ROOT)
if not runs:
    raise FileNotFoundError(f'No trials in {ROOT}; collect data or change ROOT.')
trials, configs = analyze.load_runs(runs)
if trials.empty:
    raise ValueError('No completed trials yet.')
assert set(trials.operator) == {'human'}, 'Select human data only.'
assert trials.task_id.nunique() == 1, 'Select one task for this pilot figure.'
assert not trials.duplicated(['method', 'depth', 'pair_id']).any(), 'Duplicate trials: check for reruns.'
assert trials[['pair_id', 'initial_state_hash']].notna().all().all(), 'Missing reset identity.'
assert trials.groupby('pair_id').initial_state_hash.nunique().max() == 1, 'Reset hashes differ.'
expected = pd.MultiIndex.from_product([['FC', 'FRS', 'FRS+F'], DEPTHS], names=['method', 'depth'])
observed = trials.groupby(['method', 'depth']).size()
assert observed.index.difference(expected).empty, 'Unexpected method/depth; check DEPTHS.'
counts = observed.reindex(expected, fill_value=0).rename('completed')
display(counts.unstack('depth'))
complete = bool((counts == TRIALS_PER_CONDITION).all())
if complete:
    pairs = [set(g.pair_id) for _, g in trials.groupby(['method', 'depth'])]
    assert all(p == pairs[0] for p in pairs), 'Conditions used different reset assignments.'
else:
    print('PARTIAL DATA: the figure will be labeled partial.')
print(f'{len(trials)} completed trials: {trials.task_description.iloc[0]}')


In [ ]:
task = int(trials.task_id.iloc[0])
title = f'Human pilot — task {task}: {trials.task_description.iloc[0]}'
if not complete:
    title += ' (partial)'
fig, ax = plt.subplots(figsize=(9, 5))
curve = analyze.plot_performance_vs_depth(trials, ax, metric='success', title=title)
fig.tight_layout()
export = ROOT / 'figures'
export.mkdir(exist_ok=True)
stem = 'success_vs_depth' if complete else 'success_vs_depth_partial'
fig.savefig(export / f'{stem}.png', dpi=220, bbox_inches='tight')
fig.savefig(export / f'{stem}.svg', bbox_inches='tight')
curve.to_csv(export / f'{stem}.csv', index=False)
plt.show()
display(curve)
print(f'Saved PNG, SVG, and CSV to {export}')


Error bars are descriptive Wilson 95% intervals. Report participant count, ten attempts per point, all depths, corruption +20°, adaptor translation angle +40°, ten-action chunks, and the common 600-step cap. Human commands receive no synthetic delay. This is an exploratory human pilot, not a confirmed population-level advantage.

The launcher accepts `ORDER` and `DEPTHS` to vary method/depth order across participants; account for practice and fatigue. The `on_target` plots in the oracle notebook need target annotations or ceiling anchors, which this pilot does not collect.